In [1]:
sample_count = 100
trial = 1

In [2]:
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()

## Llama 7B

In [3]:
from langchain.chains import LLMChain

# llm
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

# Prompt
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

# Parser
from langchain_core.output_parsers import StrOutputParser

In [4]:
# Prompt for LLM to do its geolocation task
prompt = PromptTemplate(
    input_variables=["headline", "body"],
    template="""<<SYS>> \n You are an assistant tasked in geo-locating this news article. \n <</SYS>> \n\n 
    [INST] Generate a SHORT response of where you think this article is talking about. BE SPECIFIC AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
    1.If there is one, the city the article is talking about. Otherwise, state that it can't be located. \n 
    2.The specific location within the city you got if you found one. \
    3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision. \
    If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. This is the article: \n\n
    Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n [/INST]""",
)


In [5]:
prompt3_1 = PromptTemplate(
    input_variables=["headline", "body"],
    template="""
    <|begin_of_text|><|start_header_id|>system<|end_header_id|>
    
    Cutting Knowledge Date: December 2023
    Today Date: 26 Jul 2024

    You are an expert in geo-location and have a deep understanding of specific places and organizations. In a short response, your task is to identify and provide the most exact real location mentioned in the news article. This can be a place, organization, facility, or any location that can help identify where the article takes place or talks about. Also, mention any specific locations or organizations explicitly found within the article that influenced your decision. If you cannot determine a location, state that explicitly. Do not discuss anthing else.<|eot_id|><|start_header_id|>user<|end_header_id|>

    You are tasked in geo-locating this news article. Generate a SHORT response specifying the most exact location you can find mentioned in the article. Give your answer in the following format:
    1. If there is one, the city the article is talking about. Otherwise, state that it can't be located. 
    2. The specific place within the city you got if you found one.
    3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision.
     
    If you cannot determine a location, state that explicitly. DO NOT MAKE UP INFORMATION. This is the news article:
    Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n <|eot_id|><|start_header_id|>assistant<|end_header_id|>""",
)


In [6]:
# Call model. Needs to be downloaded
llama_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"
llama2_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"
llama3_1_model_path = "./models/llama_3_1_8B/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"


In [ ]:
llm2 = LlamaCpp(
    model_path=llama2_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
)
output_parser = StrOutputParser()

In [ ]:
llm3_1 = LlamaCpp(
    model_path=llama3_1_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
)
output_parser = StrOutputParser()

In [9]:
chain2 = prompt | llm2 | output_parser
chain3_1 = prompt3_1 | llm3_1 | output_parser

In [10]:
# Run LLM on a given article
def run_llm2(headline, body):
    return chain2.invoke({"headline": headline, "body": body})

def run_llm3_1(headline, body):
    return chain3_1.invoke({"headline": headline, "body": body})


## NER Model

In [11]:
import spacy
from span_marker import SpanMarkerModel

In [12]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

## Google Maps

In [13]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [14]:
import ast
import requests
import googlemaps
from mapbox import Geocoder

In [15]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [16]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            return longitude, latitude
        else:
            return None
    except Exception as error:
        print(error)
        return None

### Functions to manage caches

In [17]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file, indent=4)

### Wrapper function to measure time taken by a given function

In [18]:
import time

def sec_to_hms(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    remaining_seconds = round(seconds % 60)
    return f"{hours:02}:{minutes:02}:{remaining_seconds:02}"

def check_time(func):
    
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        total_time = end_time - start_time
        total_time_formatted = sec_to_hms(total_time)
        print(f"Time taken: {total_time_formatted}")
        return result
    return wrapper

## Pipeline Entry Point

In [19]:
sample_data_path = "./sample_data/Articles_Nov_2020_March_2023.csv" # Using this as I don't have the other one

In [20]:
# Temporary. Use given article data set. Comment out when obtain the other data sate
full_df = pd.read_csv(sample_data_path)

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]

# Pick a random sample of 5 articles
raw_df = full_df.sample(sample_count)
# raw_df = full_df
len(raw_df)


100

In [ ]:
raw_df.head(10)

The ML Model honestly just needs the `id`, `header`, and `body`.

In [22]:
df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

Remove Duplicates (if any)

In [23]:
duplicates = df.duplicated(subset=['hl1'])

In [24]:
print(duplicates.value_counts())

False    100
Name: count, dtype: int64


In [25]:
df = df.drop_duplicates(subset=['hl1'])

Clean the HTML in the body and header

In [26]:
# Function to extract the text from the html of the article
func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
df['body'] = df['body'].progress_apply(func_clean_html)
df['hl1'] = df['hl1'].progress_apply(func_clean_html)

  0%|          | 0/100 [00:00<?, ?it/s]C:\Users\axel0\AppData\Local\Temp\ipykernel_28408\1969457438.py:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
100%|██████████| 100/100 [00:00<00:00, 38739.30it/s]


Clean the Body and Header with Regex

In [27]:
# Function to remove extra symbols from the text
func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
df['body'] = df['body'].progress_apply(func_clean_regex)
df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

100%|██████████| 100/100 [00:00<00:00, 108942.96it/s]


### Benchmark total times

In [28]:
time_df = pd.DataFrame(columns=['NER', 'LLM2', 'LLM3.1'])

### Explicit Article Mentions

Load the well-known locations, organizations, and neighborhoods dictionary and the locations that we don't want to allow (i.e. Too broad or incorrect ones like "Boston", "Massachussets", etc.)

In [29]:
known_title_locs_path = "./geodata/known_locs.json"
known_title_locs = load_cache(known_title_locs_path)

unwanted_entities_path = "./geodata/unwanted_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [30]:
# If a location is in the title, use that as the article's location
def explicit_filtering(header):
    # Look through the header for known locations
    locations_list = []
    lowercase_header = header.lower()
    for location in known_title_locs.keys():
        if (location.lower() in lowercase_header):
            if location not in unwanted_entities["FAC"]:
                locations_list.append(location)
    
    print(f"\nLocations found on title: \n{locations_list} \n")
    return locations_list

In [31]:
# df["Explicit_Pass"] = df["hl1"].progress_apply(explicit_filtering)
df["Explicit_Pass"] = None


In [32]:
df["Explicit_Pass"].value_counts().head(10)

Series([], Name: count, dtype: int64)

### NER Code First Pass

In [33]:
# Return all valid facilities and organizations found
def get_valid_entities(entities):
    valid_facilities = []
    valid_orgs = []

    for entity in entities:
        if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
            valid_facilities.append(entity.text)
        
        if (entity.label_ == "ORG" and entity.text not in unwanted_entities["ORG"]):
            valid_orgs.append(entity.text)
    
    valid_entities = valid_facilities + valid_orgs

    if (len(valid_entities) == 0):
        return None
    else:
        return valid_entities
        

In [34]:
# Run NER on the body of the article and return first valid facility
def run_NER(text):
    try:
        if (text == None or text == ""):
            return None
        
        entities = nlp(text).ents
        valid_entities = get_valid_entities(entities)
        return valid_entities
        
    except Exception as error:
        print(error)
        return None

In [35]:
# Chunk processing - split the article into chunks of text and run NER on each chunk
chunk_size = 100
def chunk_processing(text, chunk_size=chunk_size, chunk_limit=None):
    # Split text into smaller chunks
    chunks = split_text_into_chunks(text, chunk_size)

    all_entities = []

    if chunk_limit is not None:
        if len(chunks) > chunk_limit:
            chunks = chunks[:chunk_limit]
    
    # Process each chunk and return if valid entities are found
    for chunk in chunks:
        result = run_NER(chunk)
        if result is not None:
             all_entities.extend(result)
    
    print(f"\nAll entities for the article: \n{all_entities} \n")
    
    if len(all_entities) == 0:
        return None
    else:
        return all_entities

# Split article text into chunks of specified size
def split_text_into_chunks(text, chunk_size=chunk_size):
    words = text.split()
    chunks = [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

In [36]:
@check_time
def handle_chunk_processing(article):
    # Filter out articles that have no text
    text = article['body']
    if (text is None or text == ""):
            return None
    try:
        return chunk_processing(text, chunk_limit=5)
    except Exception as error:
        print(error)

In [37]:
start_time = time.time()

df["NER_Pass"] = df.progress_apply(handle_chunk_processing, axis=1)
# df["NER_Pass"] = None

end_time = time.time()
total_time = end_time - start_time
total_time_formatted = sec_to_hms(total_time)
print(f"Time taken: {total_time_formatted}")
time_df.loc[0, "NER"] = total_time_formatted


  2%|▏         | 2/100 [00:27<22:20, 13.68s/it]


All entities for the article: 
['GBH'] 

Time taken: 00:00:27


  3%|▎         | 3/100 [02:23<1:30:55, 56.24s/it]


All entities for the article: 
['the Mystic Aquarium', 'North Shore N.E. Aquarium', 'Blurb.com', 'Sweetwater Co.'] 

Time taken: 00:01:56


  4%|▍         | 4/100 [02:35<1:03:57, 39.97s/it]

...


All entities for the article: 
[] 

Time taken: 00:00:02


100%|██████████| 100/100 [1:54:29<00:00, 56.48s/it]


All entities for the article: 
['Rutgers University Eagleton Institute of Politics', 'the Center for Women and Politics', 'Rutgers University', 'White House', 'FiveThirtyEight', 'M.I.T.', 'the White House', 'NPR'] 

Time taken: 00:01:41


100%|██████████| 100/100 [1:54:43<00:00, 68.84s/it]


All entities for the article: 
[] 

Time taken: 00:00:14
Time taken: 01:54:44


In [38]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass
6446,0000017d-2a0a-d0ab-a17d-6aff429b0001,Wednesday November 17,Take seat on the ultimate thrill ride to explo...,None,[GBH]
11039,00000184-85a8-d006-a5dd-c5e9afa10001,Shoebert of Shoe Pond Beverly favorite seal in...,When gray seal named Shoebert appeared in Beve...,None,"[the Mystic Aquarium, North Shore N.E. Aquariu..."
626,00000176-00b0-d45d-a377-3ab9b0150001,Sunday November 29,On the eve of its 50th anniversary year celebr...,None,"[PBS, ITV plc, ITV Global Entertainment Ltd]"
10585,00000183-cc90-d9b5-ab83-ced73fa20001,Biden marijuana pardon hugely significant expe...,Last week President Joe Biden issued an execut...,None,"[the Parabola Center, Treez of Lyfe]"
3785,00000179-5b82-df8c-ad7d-db8231bd0001,Wednesday May 12,NOVA explores barriers to fertility from the s...,None,"[NOVA, NOVA, Shutterstock, bezikus Eky Studio]"
11894,00000185-ef27-dedc-afd5-ef67ae6f0001,Workforce shortages are at crisis point Healey...,Gov. Maura Healey recognizes that Massachusett...,None,"[Associated, Newton Marriott, AIM, MassReconnect]"
2067,00000177-6d8e-d20b-adff-fdcec34a0001,Consulting Giant McKinsey To Settle Opioid Cla...,McKinsey Company has reached $573 million sett...,None,"[McKinsey Company, McKinsey, NPR, McKinsey, Pu..."
8317,0000017f-f5b9-d150-a9ff-f5b921530000,Apr. 14th New England Conservatory Fellowship ...,New England Conservatory Fellowship String Qua...,None,"[GBH Studio, the GBH Studio, the Boston Public..."
10597,00000183-d0c9-d0d0-adfb-dded85fd0002,Many incomes can keep up with inflation. Now o...,Tulsa retiree Lynn Christophersen relies almos...,None,"[Social Security, the Energy Department, Socia..."
9695,00000182-367f-d6aa-a7a3-7f7fc43d0001,Can people injured on the MBTA sue to the,After series of accidents on the MBTA includin...,None,"[MBTA, the Orange Line, GBH News, Northeastern..."


### Llama Prediction

In [39]:
def filter_llama_output(log):
    # Define regex patterns to match the lines we want to remove
    llama_print_timings_pattern = re.compile(r'llama_print_timings:.*')
    llama_generate_pattern = re.compile(r'Llama.generate:.*')

    lines = log.split('\n')

    filtered_lines = []

    for line in lines:
        # If the line matches any of the unwanted patterns, skip it
        if llama_print_timings_pattern.match(line) or llama_generate_pattern.match(line):
            continue

        filtered_lines.append(line.strip())

    # Join the filtered lines back into a single string
    filtered_log = '\n'.join(filtered_lines)
    
    return filtered_log

In [40]:
# Run LLM model on the articles, and then run the NER on the prediction.
@check_time
def predict_llama2(article):
    try:
        truncated_text = article['body'][:4000]
        llama_prediction = run_llm2(article['hl1'], truncated_text)
        cleaned_prediction = filter_llama_output(llama_prediction)
        print(f"\nLlama 2 Prediction: \n{cleaned_prediction} \n")
        
        valid_entities = run_NER(cleaned_prediction)
        print(f"\nAll entities for the article from LLM 2: \n{valid_entities} \n")
        return valid_entities
    
    except Exception as error:
        print(error)
        return None

In [41]:
start_time = time.time()

df['LLM_2_Pass'] = df.progress_apply(predict_llama2, axis=1)
# df['LLM_2_Pass'] = None

end_time = time.time()
total_time = end_time - start_time
total_time_formatted = sec_to_hms(total_time)
print(f"Total time taken: {total_time_formatted}")
time_df.loc[0, "LLM2"] = total_time_formatted

  0%|          | 0/100 [00:00<?, ?it/s]
llama_print_timings:        load time =   40618.40 ms
llama_print_timings:      sample time =      84.52 ms /   247 runs   (    0.34 ms per token,  2922.56 tokens per second)
llama_print_timings: prompt eval time =   40616.66 ms /   418 tokens (   97.17 ms per token,    10.29 tokens per second)
llama_print_timings:        eval time =   58100.12 ms /   246 runs   (  236.18 ms per token,     4.23 tokens per second)
llama_print_timings:       total time =   99336.08 ms /   664 tokens



Llama 2 Prediction: 
Based on the information provided in the article, here is my response:
1. The city the article is talking about is likely to be Boston, Massachusetts, as the article mentions "GBH" (Granite Broadcasting Holdings), which is a television station based in Boston.
2. Within Boston, the specific location where the black holes are being discussed is likely to be the Harvard-Smithsonian Center for Astrophysics, which is a research center located in Cambridge, Massachusetts, just outside of Boston. The article mentions "new generation of high energy telescopes" that are being used to study black holes, which suggests that the research is being conducted at this location.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision include:
* Harvard-Smithsonian Center for Astrophysics (Cambridge, Massachusetts)
* GBH (Granite Broadcasting Holdings), a television station based in Boston.

In conclusion, based on the in

  2%|▏         | 2/100 [02:23<1:57:17, 71.81s/it]


All entities for the article from LLM 2: 
['the Harvard-Smithsonian Center for Astrophysics', 'GBH', 'Granite Broadcasting Holdings)', 'the Harvard-Smithsonian Center for Astrophysics', 'Harvard-Smithsonian Center for Astrophysics', 'GBH', 'Granite Broadcasting Holdings'] 

Time taken: 00:02:24


Llama.generate: prefix-match hit

llama_print_timings:        load time =   40618.40 ms
llama_print_timings:      sample time =      84.33 ms /   246 runs   (    0.34 ms per token,  2917.18 tokens per second)
llama_print_timings: prompt eval time =   72350.45 ms /   855 tokens (   84.62 ms per token,    11.82 tokens per second)
llama_print_timings:        eval time =   45556.32 ms /   245 runs   (  185.94 ms per token,     5.38 tokens per second)
llama_print_timings:       total time =  118360.11 ms /  1100 tokens


...


All entities for the article from LLM 2: 
['Rutgers University', 'Eagleton Institute of Politics'] 

Time taken: 00:02:48


Llama.generate: prefix-match hit

llama_print_timings:        load time =   40618.40 ms
llama_print_timings:      sample time =      68.96 ms /   210 runs   (    0.33 ms per token,  3045.33 tokens per second)
llama_print_timings: prompt eval time =   11676.60 ms /   143 tokens (   81.65 ms per token,    12.25 tokens per second)
llama_print_timings:        eval time =   33336.97 ms /   209 runs   (  159.51 ms per token,     6.27 tokens per second)
llama_print_timings:       total time =   45340.00 ms /   352 tokens



Llama 2 Prediction: 
Based on the information provided in the article, my best guess for the location is Mompox, Colombia. Here are the reasons why:
1. The article explicitly mentions "Mompox" as the hometown of Mama Icha, and it is clear that this is the place she has always called home despite living in the United States for decades.
2. Within the article, there is a mention of "La Casa de Mama Icha," which suggests that Mama Icha has a specific house or location in Mompox that holds sentimental value for her.
3. The article also mentions that Mama Icha boards a plane and flies back to Colombia at the end of her life, which further supports the idea that she is returning to her hometown of Mompox.
Therefore, based on the information provided in the article, my best guess is that Mama Icha's story is set in Mompox, Colombia. 



100%|██████████| 100/100 [3:24:43<00:00, 122.84s/it]


All entities for the article from LLM 2: 
None 

Time taken: 00:01:17
Total time taken: 03:24:44


In [42]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,LLM_2_Pass
6446,0000017d-2a0a-d0ab-a17d-6aff429b0001,Wednesday November 17,Take seat on the ultimate thrill ride to explo...,None,[GBH],[the Harvard-Smithsonian Center for Astrophysi...
11039,00000184-85a8-d006-a5dd-c5e9afa10001,Shoebert of Shoe Pond Beverly favorite seal in...,When gray seal named Shoebert appeared in Beve...,None,"[the Mystic Aquarium, North Shore N.E. Aquariu...","[Mystic Aquarium, Beverly Police]"
626,00000176-00b0-d45d-a377-3ab9b0150001,Sunday November 29,On the eve of its 50th anniversary year celebr...,None,"[PBS, ITV plc, ITV Global Entertainment Ltd]","[GBH 2, BBC Two, ITV plc, ITV Global Entertain..."
10585,00000183-cc90-d9b5-ab83-ced73fa20001,Biden marijuana pardon hugely significant expe...,Last week President Joe Biden issued an execut...,None,"[the Parabola Center, Treez of Lyfe]","[the Parabola Center, the Parabola Center, the..."
3785,00000179-5b82-df8c-ad7d-db8231bd0001,Wednesday May 12,NOVA explores barriers to fertility from the s...,None,"[NOVA, NOVA, Shutterstock, bezikus Eky Studio]","[GBH 2, GBH 2]"
11894,00000185-ef27-dedc-afd5-ef67ae6f0001,Workforce shortages are at crisis point Healey...,Gov. Maura Healey recognizes that Massachusett...,None,"[Associated, Newton Marriott, AIM, MassReconnect]","[the Associated Industries of Massachusetts, A..."
2067,00000177-6d8e-d20b-adff-fdcec34a0001,Consulting Giant McKinsey To Settle Opioid Cla...,McKinsey Company has reached $573 million sett...,None,"[McKinsey Company, McKinsey, NPR, McKinsey, Pu...",[McKinsey Company]
8317,0000017f-f5b9-d150-a9ff-f5b921530000,Apr. 14th New England Conservatory Fellowship ...,New England Conservatory Fellowship String Qua...,None,"[GBH Studio, the GBH Studio, the Boston Public...","[the GBH Studio, GBH Studio, the Boston Public..."
10597,00000183-d0c9-d0d0-adfb-dded85fd0002,Many incomes can keep up with inflation. Now o...,Tulsa retiree Lynn Christophersen relies almos...,None,"[Social Security, the Energy Department, Socia...","[Social Security, The Energy Department]"
9695,00000182-367f-d6aa-a7a3-7f7fc43d0001,Can people injured on the MBTA sue to the,After series of accidents on the MBTA includin...,None,"[MBTA, the Orange Line, GBH News, Northeastern...","[the Orange Line, MBTA, MBTA, Orange Line, the..."


In [43]:
@check_time
def predict_llama3_1(article):
    try:
        truncated_text = article['body'][:4000]
        llama_prediction = run_llm3_1(article['hl1'], truncated_text)
        cleaned_prediction = filter_llama_output(llama_prediction)
        print(f"\nLlama 3.1 Prediction: \n{cleaned_prediction} \n")

        valid_entities = run_NER(cleaned_prediction)
        print(f"\nAll entities for the article from LLM 3.1: \n{valid_entities} \n")
        return valid_entities
    
    except Exception as error:
        print(error)
        return None

In [44]:
start_time = time.time()

df['LLM_3_1_Pass'] = df.progress_apply(predict_llama3_1, axis=1)

end_time = time.time()
total_time = end_time - start_time
total_time_formatted = sec_to_hms(total_time)
print(f"Total time taken: {total_time_formatted}")
time_df.loc[0, "LLM3.1"] = total_time_formatted

  0%|          | 0/100 [00:00<?, ?it/s]
llama_print_timings:        load time =   45250.18 ms
llama_print_timings:      sample time =      55.90 ms /    49 runs   (    1.14 ms per token,   876.58 tokens per second)
llama_print_timings: prompt eval time =   45236.70 ms /   449 tokens (  100.75 ms per token,     9.93 tokens per second)
llama_print_timings:        eval time =    9032.87 ms /    48 runs   (  188.18 ms per token,     5.31 tokens per second)
llama_print_timings:       total time =   54583.67 ms /   497 tokens



Llama 3.1 Prediction: 


1. It can't be located.
2. No specific place within a city is mentioned.
3. The article discusses the concept of black holes, galaxies, and other celestial objects, but does not mention any specific location or organization. 



  2%|▏         | 2/100 [01:10<57:40, 35.31s/it]


All entities for the article from LLM 3.1: 
None 

Time taken: 00:01:11


Llama.generate: prefix-match hit

llama_print_timings:        load time =   45250.18 ms
llama_print_timings:      sample time =      70.43 ms /    62 runs   (    1.14 ms per token,   880.32 tokens per second)
llama_print_timings: prompt eval time =   64884.27 ms /   730 tokens (   88.88 ms per token,    11.25 tokens per second)
llama_print_timings:        eval time =   11861.33 ms /    61 runs   (  194.45 ms per token,     5.14 tokens per second)
llama_print_timings:       total time =   76982.10 ms /   791 tokens


...


All entities for the article from LLM 3.1: 
['Rutgers University', 'Eagleton Institute of Politics', 'Center for Women and Politics', 'Rutgers University'] 

Time taken: 00:01:54


Llama.generate: prefix-match hit

llama_print_timings:        load time =   45250.18 ms
llama_print_timings:      sample time =     156.98 ms /   153 runs   (    1.03 ms per token,   974.62 tokens per second)
llama_print_timings: prompt eval time =    9905.66 ms /   120 tokens (   82.55 ms per token,    12.11 tokens per second)
llama_print_timings:        eval time =   26389.74 ms /   152 runs   (  173.62 ms per token,     5.76 tokens per second)
llama_print_timings:       total time =   37012.88 ms /   272 tokens



Llama 3.1 Prediction: 


1. The city mentioned in the article is: Mompox. However, since Mompox was a colonial town that existed until it was moved to Leticia and renamed in 1974, I will identify its exact real-time location which would be Leticia. However for legacy purposes, we will refer to the original Mompox.

2. The specific place within the city of Leticia is: None explicitly mentioned but implied to be near or within the boundaries of Leticia since Mama Icha was returning back home.

3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced my decision are: Mompox (now known as Leticia), the United States, and Colombia. 



100%|██████████| 100/100 [1:58:50<00:00, 71.30s/it]


All entities for the article from LLM 3.1: 
['Mompox', 'Leticia'] 

Time taken: 00:01:07
Total time taken: 01:58:50


In [45]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,LLM_2_Pass,LLM_3_1_Pass
6446,0000017d-2a0a-d0ab-a17d-6aff429b0001,Wednesday November 17,Take seat on the ultimate thrill ride to explo...,None,[GBH],[the Harvard-Smithsonian Center for Astrophysi...,None
11039,00000184-85a8-d006-a5dd-c5e9afa10001,Shoebert of Shoe Pond Beverly favorite seal in...,When gray seal named Shoebert appeared in Beve...,None,"[the Mystic Aquarium, North Shore N.E. Aquariu...","[Mystic Aquarium, Beverly Police]","[Blurb.com, Sweetwater Co.]"
626,00000176-00b0-d45d-a377-3ab9b0150001,Sunday November 29,On the eve of its 50th anniversary year celebr...,None,"[PBS, ITV plc, ITV Global Entertainment Ltd]","[GBH 2, BBC Two, ITV plc, ITV Global Entertain...","[ITV plc, ITV Global Entertainment Ltd, PBS]"
10585,00000183-cc90-d9b5-ab83-ced73fa20001,Biden marijuana pardon hugely significant expe...,Last week President Joe Biden issued an execut...,None,"[the Parabola Center, Treez of Lyfe]","[the Parabola Center, the Parabola Center, the...","[The Parabola Center, Treez of Lyfe]"
3785,00000179-5b82-df8c-ad7d-db8231bd0001,Wednesday May 12,NOVA explores barriers to fertility from the s...,None,"[NOVA, NOVA, Shutterstock, bezikus Eky Studio]","[GBH 2, GBH 2]","[GBH, GBH, PBS]"
11894,00000185-ef27-dedc-afd5-ef67ae6f0001,Workforce shortages are at crisis point Healey...,Gov. Maura Healey recognizes that Massachusett...,None,"[Associated, Newton Marriott, AIM, MassReconnect]","[the Associated Industries of Massachusetts, A...","[Newton Marriott, Associated Industries of Mas..."
2067,00000177-6d8e-d20b-adff-fdcec34a0001,Consulting Giant McKinsey To Settle Opioid Cla...,McKinsey Company has reached $573 million sett...,None,"[McKinsey Company, McKinsey, NPR, McKinsey, Pu...",[McKinsey Company],"[Johnson & Johnson, McKesson, Walmart]"
8317,0000017f-f5b9-d150-a9ff-f5b921530000,Apr. 14th New England Conservatory Fellowship ...,New England Conservatory Fellowship String Qua...,None,"[GBH Studio, the GBH Studio, the Boston Public...","[the GBH Studio, GBH Studio, the Boston Public...","[GBH Studio, the Boston Public Library, GBH St..."
10597,00000183-d0c9-d0d0-adfb-dded85fd0002,Many incomes can keep up with inflation. Now o...,Tulsa retiree Lynn Christophersen relies almos...,None,"[Social Security, the Energy Department, Socia...","[Social Security, The Energy Department]","[Social Security, The Energy Department, The C..."
9695,00000182-367f-d6aa-a7a3-7f7fc43d0001,Can people injured on the MBTA sue to the,After series of accidents on the MBTA includin...,None,"[MBTA, the Orange Line, GBH News, Northeastern...","[the Orange Line, MBTA, MBTA, Orange Line, the...","[MBTA, MBTA]"


In [46]:
print(df['Explicit_Pass'].value_counts().sum())
df['Explicit_Pass'].value_counts()

0


Series([], Name: count, dtype: int64)

In [47]:
print(df['NER_Pass'].value_counts().sum())
df['NER_Pass'].value_counts()

93


NER_Pass
[GBH]                                                                                                                                                                                                                                                                                                                                                                                                                                                                       1
[Massachusetts Water Resources Authority, Biobot Analytics, MWRA, Tufts Medical Center, the Harvard T.H. Chan School of Public Health, GBH News]                                                                                                                                                                                                                                                                                                                            1
[MBTA, the Federal Transit Administration, FTA, MBTA, Orange Line, 

In [48]:
print(df['LLM_2_Pass'].value_counts().sum())
df['LLM_2_Pass'].value_counts()

90


LLM_2_Pass
[the Harvard-Smithsonian Center for Astrophysics, GBH, Granite Broadcasting Holdings), the Harvard-Smithsonian Center for Astrophysics, Harvard-Smithsonian Center for Astrophysics, GBH, Granite Broadcasting Holdings]    1
[The Boston Common, Peter Park, the Boston Common, Peter Park, Cherokee County Sheriff Office]                                                                                                                              1
[The Operations Control Center, the Operations Control Center, MBTA, Massachusetts Bay Transportation Authority, FTA, MBTA, FTA, MBTA]                                                                                      1
[the American Academy of Pediatrics, the Children Hospital Association, American Academy of Pediatrics, AAP, Children Hospital Association, CHA, Centers for Disease Control and Prevention, CDC]                           1
[MBTA, MBTA, Massachusetts Bay Transportation Authority, Harvard Law School, Massachusetts Cultural C

In [49]:
print(df['LLM_3_1_Pass'].value_counts().sum())
df['LLM_3_1_Pass'].value_counts()

83


LLM_3_1_Pass
[Blurb.com, Sweetwater Co.]                                                                                                                                                                              1
[Coolidge Corner Theatre, Coolidge Corner Theatre, Disney, Bay Windows, South End News]                                                                                                                  1
[Operations Control Center\n3, Tufts New England Medical Center, MBTA, MBTA, Massachusetts Bay Transportation Authority, Federal Transit Administration, FTA]                                            1
[American Academy of Pediatrics, AAP, Children Hospital Association, Centers for Disease Control and Prevention, CDC]                                                                                    1
[Harvard Law School, Boston Public Radio, Massachusetts Bay Transportation Authority, MBTA, The NAACP Advocacy and Policy Committee, The Mass League of Community Health Center

In [50]:
len(df)

100

In [51]:
df.to_csv(f"./results/benchmarking_{sample_count}_samples_trial_{trial}.csv")
time_df.to_csv(f"./results/benchmarking_times_{sample_count}_samples_trial_{trial}.csv")

In [52]:
print(df.count())
print(df.dropna(subset=['NER_Pass'])["LLM_2_Pass"].notnull().sum())
print(df.dropna(subset=['NER_Pass'])["LLM_3_1_Pass"].notnull().sum())

print(df.dropna(subset=['LLM_2_Pass'])["LLM_3_1_Pass"].notnull().sum())

_id              100
hl1              100
body             100
Explicit_Pass      0
NER_Pass          93
LLM_2_Pass        90
LLM_3_1_Pass      83
dtype: int64
88
81
79


Extract locations from the most specific pass

In [93]:
# Get the locations from the most specific pass for a given article
def extractLocations(article):
    for key in ['Explicit_Pass', 'NER_Pass', 'LLM_2_Pass', 'LLM_3_1_Pass']:
        location = article.get(key)
        if location is not None:
            return location
    return None

In [94]:
# df['Locations'] = df.progress_apply(extractLocations, axis=1)

In [95]:
from fuzzywuzzy import fuzz

def normalize_location(location):
    location = location.lower().strip()
    if location.startswith("the "):
        location = location[4:]
    return location

def are_same_location(loc1, loc2, threshold=85):
    norm_loc1 = normalize_location(loc1)
    norm_loc2 = normalize_location(loc2)
    similarity = fuzz.token_set_ratio(norm_loc1, norm_loc2)
    return similarity >= threshold

def extractAllLocations(article):
    locations_list = []
    for key in ['Explicit_Pass', 'NER_Pass', 'LLM_2_Pass', 'LLM_3_1_Pass']:
        location = article.get(key)
        if location is not None:
            locations_list.extend(location)
    
    if len(locations_list) == 0:
        return None
    else:
        # Ensure unique locations considering variations
        unique_locations = []
        for loc in locations_list:
            if not any(are_same_location(loc, unique_loc) for unique_loc in unique_locations):
                unique_locations.append(loc)
        return unique_locations

In [96]:
df['Locations'] = df.progress_apply(extractAllLocations, axis=1)

100%|██████████| 100/100 [00:00<00:00, 1640.92it/s]


In [ ]:
df.head(10)

## Get the coordinates

In [98]:
known_locations_path = "./geodata/known_locations.json"  
known_locations = load_cache(known_locations_path)

In [99]:
# Get the coordinates of the location
def getCoordinates(location): 
    if (location == None or len(location) == 0): return None  
    
    # Only get coordinates if the location is not already known
    if (location in known_locations):
        longitude, latitude = known_locations[location]["coordinates"]
    else:
        # Get coordinates and save to cache
        longitude, latitude = callGoogleMapsAPI(location)
        known_locations[location] = {"coordinates": [longitude, latitude], "tract": None, "county": None}
        save_cache_to_file(known_locations, known_locations_path)

    return [longitude, latitude]

In [100]:
def getAllCoordinates(locations):
    coordinates_list = []

    if (locations == None): return None
    
    for location in locations:
        coordinates = getCoordinates(location)
        if coordinates is not None:
            coordinates_list.append(coordinates)
    return coordinates_list

In [101]:
df['Coordinates'] = df['Locations'].progress_apply(getAllCoordinates)

100%|██████████| 100/100 [00:00<00:00, 100078.84it/s]


In [ ]:
df.head(10)

## Geocode locations

In [103]:
# Get the census tract of the location
def query_census_api(location, coordinates):
    longitude, latitude = coordinates
    base_url = f'https://geocoding.geo.census.gov/geocoder/geographies/coordinates?'
    survey_ver = f'&benchmark=4&vintage=4&layers=2020 Census Blocks&format=json'
    url = f'{base_url}x={longitude}&y={latitude}{survey_ver}'

    response = requests.get(url)

    # Check if response is valid
    if (response.status_code == 200):
        results = response.json()
        try:
            tract = results['result']['geographies']['2020 Census Blocks'][0]['TRACT']
            county = results['result']['geographies']['2020 Census Blocks'][0]['COUNTY']

            return tract, county
        except IndexError:
            print("Unable to retrieve census geography for: " + location)
        except KeyError:
            print("Location is outside of the United States: " + location)
        except Exception as error:
            print(error)

    return None, None  # Return this if API call failed or no tracts found

In [104]:
# Get the census tract and county of the location
def geocode(location):
    if (location is None or len(location) == 0): return None, None  

    # Only geocode if it's not known
    Tract = known_locations[location]["tract"]
    County = known_locations[location]["county"]
    if (Tract is None or County is None):
        # Geocode article
        coordinates = known_locations[location]["coordinates"]
        Tract, County = query_census_api(location, coordinates)

        # Save to cache
        known_locations[location]["tract"] = Tract
        known_locations[location]["county"] = County
        save_cache_to_file(known_locations, known_locations_path)
    
    return Tract, County

In [105]:
def getAllGeocodes(locations):
    tracts = []
    counties = []

    if (locations == None): return None, None
    
    for location in locations:
        Tract, County = geocode(location)
        tracts.append(Tract)
        counties.append(County)

    return tracts, counties

In [106]:
df[['Tracts', 'Counties']] = df['Locations'].progress_apply(getAllGeocodes).apply(pd.Series)


100%|██████████| 100/100 [00:00<00:00, 50081.24it/s]


In [107]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,LLM_2_Pass,LLM_3_1_Pass,Locations,Coordinates,Tracts,Counties,Neighborhoods
6446,0000017d-2a0a-d0ab-a17d-6aff429b0001,Wednesday November 17,Take seat on the ultimate thrill ride to explo...,None,[GBH],[the Harvard-Smithsonian Center for Astrophysi...,None,"[GBH, the Harvard-Smithsonian Center for Astro...","[[-71.3824374, 42.4072107], [-71.1280685, 42.3...","[365100, 354500, 365100]","[017, 017, 017]","[Cambridge, Cambridge, Cambridge]"
11039,00000184-85a8-d006-a5dd-c5e9afa10001,Shoebert of Shoe Pond Beverly favorite seal in...,When gray seal named Shoebert appeared in Beve...,None,"[the Mystic Aquarium, North Shore N.E. Aquariu...","[Mystic Aquarium, Beverly Police]","[Blurb.com, Sweetwater Co.]","[the Mystic Aquarium, North Shore N.E. Aquariu...","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100, 365100, 971600, 217300]","[017, 017, 017, 037, 009]","[Cambridge, Cambridge, Cambridge]"
626,00000176-00b0-d45d-a377-3ab9b0150001,Sunday November 29,On the eve of its 50th anniversary year celebr...,None,"[PBS, ITV plc, ITV Global Entertainment Ltd]","[GBH 2, BBC Two, ITV plc, ITV Global Entertain...","[ITV plc, ITV Global Entertainment Ltd, PBS]","[PBS, ITV plc, ITV Global Entertainment Ltd, G...","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100, 365100, 704201, 365100]","[017, 017, 017, 027, 017]","[Cambridge, Cambridge, Cambridge, Cambridge]"
10585,00000183-cc90-d9b5-ab83-ced73fa20001,Biden marijuana pardon hugely significant expe...,Last week President Joe Biden issued an execut...,None,"[the Parabola Center, Treez of Lyfe]","[the Parabola Center, the Parabola Center, the...","[The Parabola Center, Treez of Lyfe]","[the Parabola Center, Treez of Lyfe]","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100]","[017, 017]","[Cambridge, Cambridge]"
3785,00000179-5b82-df8c-ad7d-db8231bd0001,Wednesday May 12,NOVA explores barriers to fertility from the s...,None,"[NOVA, NOVA, Shutterstock, bezikus Eky Studio]","[GBH 2, GBH 2]","[GBH, GBH, PBS]","[NOVA, Shutterstock, bezikus Eky Studio, GBH 2...","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100, 365100, 704201, 365100]","[017, 017, 017, 027, 017]","[Cambridge, Cambridge, Cambridge, Cambridge]"
11894,00000185-ef27-dedc-afd5-ef67ae6f0001,Workforce shortages are at crisis point Healey...,Gov. Maura Healey recognizes that Massachusett...,None,"[Associated, Newton Marriott, AIM, MassReconnect]","[the Associated Industries of Massachusetts, A...","[Newton Marriott, Associated Industries of Mas...","[Associated, Newton Marriott, AIM, MassReconnect]","[[-71.3824374, 42.4072107], [-71.2578827, 42.3...","[365100, 374700, 365100, 365100]","[017, 017, 017, 017]","[Cambridge, Cambridge, Cambridge]"
2067,00000177-6d8e-d20b-adff-fdcec34a0001,Consulting Giant McKinsey To Settle Opioid Cla...,McKinsey Company has reached $573 million sett...,None,"[McKinsey Company, McKinsey, NPR, McKinsey, Pu...",[McKinsey Company],"[Johnson & Johnson, McKesson, Walmart]","[McKinsey Company, NPR, Purdue Pharma, Johnson...","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100, 365100, 365100, 365100, 36510...","[017, 017, 017, 017, 017, 017, 017]","[Cambridge, Cambridge, Cambridge, Cambridge, C..."
8317,0000017f-f5b9-d150-a9ff-f5b921530000,Apr. 14th New England Conservatory Fellowship ...,New England Conservatory Fellowship String Qua...,None,"[GBH Studio, the GBH Studio, the Boston Public...","[the GBH Studio, GBH Studio, the Boston Public...","[GBH Studio, the Boston Public Library, GBH St...","[GBH Studio, the Boston Public Library, the Ne...","[[-71.3824374, 42.4072107], [-71.0788285, 42.3...","[365100, 010600, 010405]","[017, 025, 025]","[Cambridge, Back Bay, Fenway]"
10597,00000183-d0c9-d0d0-adfb-dded85fd0002,Many incomes can keep up with inflation. Now o...,Tulsa retiree Lynn Christophersen relies almos...,None,"[Social Security, the Energy Department, Socia...","[Social Security, The En

## Get Neighborhoods

In [108]:
original_neigh_tract_dict = {
	"Fenway" : ["010103", "010104", "010204", "010408", "010404", "010403", "981501", "010405", "010206", "010205"],
	"Downtown": ["030302", "070202", "070102", "030301", "070104", "070103", "070201"],
	"Beacon Hill": ["020200", "020302", "020101", "981700"],
	"Dorchester" : [
	"092400", "091400", "090300", "091800", "092300", "100601", "090901", 
	"100400", "090100", "091001","090200", "100200", "091700", "092200", "090700",
	"091500", "091300", "100300", "100100", "092000", "100500", "100800", "100603",
	"091200", "100700", "092101", "091900", "091600", "091100"
	],
	"Mattapan": ["100900", "101002", "101102", "981100", "101001","101101"],
	"Jamaica Plain": [
	"120103", "981800", "110105", "120600", "120700", "120301", "081200", "120105","081101",
	"981000", "120500", "120104", "120201", "110106", "081301", "120400"
	],
	"Roslindale": ["110502", "110104", "110501", "110401", "140106", "110301", "110607", "110403","110201"],
	"Roxbury": [
	"081500", "080500", "070801", "080100", "081800", "980300", "082000", "080601", "081700", "080300",
	"090600", "081400", "090400", "070901", "082100", "081900", "081302","080401"
	],
	"West End": ["020304", "020301", "020305"],
	"Longwood": ["010300", "081001"],
	"South Boston": ["061101", "060700", "060101", "061201", "061000", "060800", "981201", "060200", "061202", "060400", "061203", "060301", "060601", "060501"],
	"Back Bay": ["010702", "010701", "010802", "010801", "010500", "010600"],
	"Charlestown": ["040100", "040300", "040401", "040600", "040801", "040200"],
	"Allston": ["000604", "000804", "000703", "000704", "000806", "000101", "000807", "000701", "000805"],
	"Hyde Park": ["140107", "140201", "140105", "980700", "140300", "140202", "140400", "140102"],
	"East Boston": ["050500", "050600", "981502", "050101", "981300", "050901", "050300", "050700", "050400", "051000", "981600", "051200", "050200", "051101"],
	"South End": ["070301", "070302", "070502", "070501", "071101", "070600", "070700", "070902", "070802", "071201", "070402"],
	"West Roxbury": ["980900", "130406", "981900", "130404", "110601", "130300", "130402", "130200", "130101"],
	"South Boston Waterfront": ["981202", "060602", "060603", "061204", "060604"],
	"North End": ["030200", "030100", "030500", "030400"],
	"Cambridge": ["354300", "354200", "353102", "353600", "352300", "354100", "359400", "353300", "353700", "353200", 
	"354601", "355000", "354602", "354000", "354901", "354902", "353900", "354700", "352102", "354500", "354800", "352600", 
	"354400", "353101", "352900", "353000", "352101", "353800", "352500", "352400", "352700", "352200", "352800"
  	],
	"Chelsea": ["160400", "160103", "160102", "160300", "160601", "160602", "160501", "160502", "160200"],
	

}

In [109]:
neigh_tract_dict = {
	"Fenway" : ["010103", "010104", "010204", "010408", "010404", "010403", "981501", "010405", "010206", "010205"],
	"Downtown": ["030302", "070202", "070102", "030301", "070104", "070103", "070201"],
	"Beacon Hill": ["020200", "020302", "020101", "981700"],
	"Dorchester" : [
	"092400", "091400", "090300", "091800", "092300", "100601", "090901", 
	"100400", "090100", "091001","090200", "100200", "091700", "092200", "090700",
	"091500", "091300", "100300", "100100", "092000", "100500", "100800", "100603",
	"091200", "100700", "092101", "091900", "091600", "091100"
	],
	"Mattapan": ["100900", "101002", "101102", "981100", "101001","101101"],
	"Jamaica Plain": [
	"120103", "981800", "110105", "120600", "120700", "120301", "081200", "120105","081101",
	"981000", "120500", "120104", "120201", "110106", "081301", "120400"
	],
	"Roslindale": ["110502", "110104", "110501", "110401", "140106", "110301", "110607", "110403","110201"],
	"Roxbury": [
	"081500", "080500", "070801", "080100", "081800", "980300", "082000", "080601", "081700", "080300",
	"090600", "081400", "090400", "070901", "082100", "081900", "081302","080401"
	],
	"West End": ["020304", "020301", "020305"],
	"Longwood": ["010300", "081001"],
	"South Boston": ["061101", "060700", "060101", "061201", "061000", "060800", "981201", "060200", "061202", "060400", "061203", "060301", "060601", "060501"],
	"Back Bay": ["010702", "010701", "010802", "010801", "010500", "010600"],
	"Charlestown": ["040100", "040300", "040401", "040600", "040801", "040200"],
	"Allston": ["000604", "000804", "000703", "000704", "000806", "000101", "000807", "000701", "000805"],
	"Hyde Park": ["140107", "140201", "140105", "980700", "140300", "140202", "140400", "140102"],
	"East Boston": ["050500", "050600", "981502", "050101", "981300", "050901", "050300", "050700", "050400", "051000", "981600", "051200", "050200", "051101"],
	"South End": ["070301", "070302", "070502", "070501", "071101", "070600", "070700", "070902", "070802", "071201", "070402"],
	"West Roxbury": ["980900", "130406", "981900", "130404", "110601", "130300", "130402", "130200", "130101"],
	"South Boston Waterfront": ["981202", "060602", "060603", "061204", "060604"],
	"North End": ["030200", "030100", "030500", "030400"],
	"Cambridge": ["354300", "354200", "353102", "353600", "352300", "354100", "359400", "353300", "353700", "353200", 
	"354601", "355000", "354602", "354000", "354901", "354902", "353900", "354700", "352102", "354500", "354800", "352600", 
	"354400", "353101", "352900", "353000", "352101", "353800", "352500", "352400", "352700", "352200", "352800", 
    "365100", "361300"
  	],
	"Chelsea": ["160400", "160103", "160102", "160300", "160601", "160602", "160501", "160502", "160200"],
	

}

In [110]:
def string_to_list(s):
    if(s != ''):
        return [s]
    else:
        return []  # Return the string as a single-element list

In [111]:
def find_neighborhood_by_tract(search_dict, tract_to_find):
	for key, values in search_dict.items():
		if (tract_to_find in values):
			return key
	return "Unknown Neighborhood"

def locateNeighborhoods(tract):
	if (tract == None):
		return None

	query = find_neighborhood_by_tract(neigh_tract_dict, tract)
	if (query != None):
		return string_to_list(query)
	else: 
		return string_to_list("None")


In [121]:
unknown_tracts_path = "./geodata/unknown_tracts.json"
unknown_tracts = load_cache(unknown_tracts_path)

In [122]:
def handle_tract_to_neighborhood(articles):
    tracts = articles['Tracts']

    neighborhoods = []

    if (tracts == None): return None

    for i, tract in enumerate(tracts):
        neighborhood = locateNeighborhoods(tract)
        if neighborhood is not None:
            if (neighborhood[0] == "Unknown Neighborhood"):
                location = articles['Locations'][i]
                if (tract not in unknown_tracts):
                    unknown_tracts[tract] = {"County": articles['Counties'][i], "Locations": [location]}
                    print(f"Unknown neighborhood for location: {location} with tract: {tract}")
                elif (location not in unknown_tracts[tract]["Locations"]):
                    unknown_tracts[tract]["Locations"].append(location)
                    print(f"Unknown neighborhood for location: {location} with tract: {tract}")
            neighborhoods.extend(neighborhood)

    save_cache_to_file(unknown_tracts, unknown_tracts_path)
    return neighborhoods

In [123]:
df["Neighborhoods"] = df.progress_apply(handle_tract_to_neighborhood, axis=1)

100%|██████████| 100/100 [00:00<00:00, 1502.04it/s]

Unknown neighborhood for location: Sweetwater Co. with tract: 971600
Unknown neighborhood for location: Beverly Police with tract: 217300
Unknown neighborhood for location: GBH 2 with tract: 704201
Unknown neighborhood for location: Newton Marriott with tract: 374700
Unknown neighborhood for location: GBH Fraser Performance Studio with tract: 000102
Unknown neighborhood for location: the Nevada Independent with tract: 960100
Unknown neighborhood for location: City Hall with tract: 500101
Unknown neighborhood for location: Worcester City Council with tract: 731700
Unknown neighborhood for location: Mission Hill street with tract: 081102
Unknown neighborhood for location: The Massachusetts Department of Elementary and Secondary Education with tract: 342402
Unknown neighborhood for location: Mass. Ave with tract: 356100
Unknown neighborhood for location: Newport with tract: 268300
Unknown neighborhood for location: Massachusetts Avenue with tract: 356100
Unknown neighborhood for location:

In [81]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,LLM_2_Pass,LLM_3_1_Pass,Locations,Coordinates,Tracts,Counties,Neighborhoods
6446,0000017d-2a0a-d0ab-a17d-6aff429b0001,Wednesday November 17,Take seat on the ultimate thrill ride to explo...,None,[GBH],[the Harvard-Smithsonian Center for Astrophysi...,None,"[GBH, the Harvard-Smithsonian Center for Astro...","[[-71.3824374, 42.4072107], [-71.1280685, 42.3...","[365100, 354500, 365100]","[017, 017, 017]","[Cambridge, Cambridge, Cambridge]"
11039,00000184-85a8-d006-a5dd-c5e9afa10001,Shoebert of Shoe Pond Beverly favorite seal in...,When gray seal named Shoebert appeared in Beve...,None,"[the Mystic Aquarium, North Shore N.E. Aquariu...","[Mystic Aquarium, Beverly Police]","[Blurb.com, Sweetwater Co.]","[the Mystic Aquarium, North Shore N.E. Aquariu...","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100, 365100, 971600, 217300]","[017, 017, 017, 037, 009]","[Cambridge, Cambridge, Cambridge, Unknown Neig..."
626,00000176-00b0-d45d-a377-3ab9b0150001,Sunday November 29,On the eve of its 50th anniversary year celebr...,None,"[PBS, ITV plc, ITV Global Entertainment Ltd]","[GBH 2, BBC Two, ITV plc, ITV Global Entertain...","[ITV plc, ITV Global Entertainment Ltd, PBS]","[PBS, ITV plc, ITV Global Entertainment Ltd, G...","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100, 365100, 704201, 365100]","[017, 017, 017, 027, 017]","[Cambridge, Cambridge, Cambridge, Unknown Neig..."
10585,00000183-cc90-d9b5-ab83-ced73fa20001,Biden marijuana pardon hugely significant expe...,Last week President Joe Biden issued an execut...,None,"[the Parabola Center, Treez of Lyfe]","[the Parabola Center, the Parabola Center, the...","[The Parabola Center, Treez of Lyfe]","[the Parabola Center, Treez of Lyfe]","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100]","[017, 017]","[Cambridge, Cambridge]"
3785,00000179-5b82-df8c-ad7d-db8231bd0001,Wednesday May 12,NOVA explores barriers to fertility from the s...,None,"[NOVA, NOVA, Shutterstock, bezikus Eky Studio]","[GBH 2, GBH 2]","[GBH, GBH, PBS]","[NOVA, Shutterstock, bezikus Eky Studio, GBH 2...","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100, 365100, 704201, 365100]","[017, 017, 017, 027, 017]","[Cambridge, Cambridge, Cambridge, Unknown Neig..."
11894,00000185-ef27-dedc-afd5-ef67ae6f0001,Workforce shortages are at crisis point Healey...,Gov. Maura Healey recognizes that Massachusett...,None,"[Associated, Newton Marriott, AIM, MassReconnect]","[the Associated Industries of Massachusetts, A...","[Newton Marriott, Associated Industries of Mas...","[Associated, Newton Marriott, AIM, MassReconnect]","[[-71.3824374, 42.4072107], [-71.2578827, 42.3...","[365100, 374700, 365100, 365100]","[017, 017, 017, 017]","[Cambridge, Unknown Neighborhood, Cambridge, C..."
2067,00000177-6d8e-d20b-adff-fdcec34a0001,Consulting Giant McKinsey To Settle Opioid Cla...,McKinsey Company has reached $573 million sett...,None,"[McKinsey Company, McKinsey, NPR, McKinsey, Pu...",[McKinsey Company],"[Johnson & Johnson, McKesson, Walmart]","[McKinsey Company, NPR, Purdue Pharma, Johnson...","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100, 365100, 365100, 365100, 36510...","[017, 017, 017, 017, 017, 017, 017]","[Cambridge, Cambridge, Cambridge, Cambridge, C..."
8317,0000017f-f5b9-d150-a9ff-f5b921530000,Apr. 14th New England Conservatory Fellowship ...,New England Conservatory Fellowship String Qua...,None,"[GBH Studio, the GBH Studio, the Boston Public...","[the GBH Studio, GBH Studio, the Boston Public...","[GBH Studio, the Boston Public Library, GBH St...","[GBH Studio, the Boston Public Library, the Ne...","[[-71.3824374, 42.4072107], [-71.0788285, 42.3...","[365100, 010600, 010405]","[017, 025, 025]","[Cambridge, Back Bay, Fenway]"
10597,00000183-d0c9-d0d0-adfb-dded85fd0002,Many incomes can keep up with inflation. Now o...,Tulsa retiree Lynn Christophersen relies almos...,None,"[Social Security, the Energy Depar

In [88]:
def remove_unknown_neighborhoods(df):
    def remove_unknowns(row):
        neighborhoods = row['Neighborhoods']
        if neighborhoods is None:
            return row
        
        indexes_to_remove = [i for i, neighborhood in enumerate(neighborhoods) if neighborhood == 'Unknown Neighborhood']
        
        for column in ['Tracts', 'Counties', 'Coordinates', 'Neighborhoods']:
            if row[column] is not None:
                row[column] = [v for i, v in enumerate(row[column]) if i not in indexes_to_remove]
        # row['Locations'] = [v for i, v in enumerate(row['Locations']) if i not in indexes_to_remove]
        
        return row

    df = df.apply(remove_unknowns, axis=1)
    return df

In [90]:
df = remove_unknown_neighborhoods(df)

In [91]:
df

,_id,hl1,body,Explicit_Pass,NER_Pass,LLM_2_Pass,LLM_3_1_Pass,Locations,Coordinates,Tracts,Counties,Neighborhoods
6446,0000017d-2a0a-d0ab-a17d-6aff429b0001,Wednesday November 17,Take seat on the ultimate thrill ride to explo...,None,[GBH],[the Harvard-Smithsonian Center for Astrophysi...,None,"[GBH, the Harvard-Smithsonian Center for Astro...","[[-71.3824374, 42.4072107], [-71.1280685, 42.3...","[365100, 354500, 365100]","[017, 017, 017]","[Cambridge, Cambridge, Cambridge]"
11039,00000184-85a8-d006-a5dd-c5e9afa10001,Shoebert of Shoe Pond Beverly favorite seal in...,When gray seal named Shoebert appeared in Beve...,None,"[the Mystic Aquarium, North Shore N.E. Aquariu...","[Mystic Aquarium, Beverly Police]","[Blurb.com, Sweetwater Co.]","[the Mystic Aquarium, North Shore N.E. Aquariu...","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100, 365100]","[017, 017, 017]","[Cambridge, Cambridge, Cambridge]"
626,00000176-00b0-d45d-a377-3ab9b0150001,Sunday November 29,On the eve of its 50th anniversary year celebr...,None,"[PBS, ITV plc, ITV Global Entertainment Ltd]","[GBH 2, BBC Two, ITV plc, ITV Global Entertain...","[ITV plc, ITV Global Entertainment Ltd, PBS]","[PBS, ITV plc, ITV Global Entertainment Ltd, G...","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100, 365100, 365100]","[017, 017, 017, 017]","[Cambridge, Cambridge, Cambridge, Cambridge]"
10585,00000183-cc90-d9b5-ab83-ced73fa20001,Biden marijuana pardon hugely significant expe...,Last week President Joe Biden issued an execut...,None,"[the Parabola Center, Treez of Lyfe]","[the Parabola Center, the Parabola Center, the...","[The Parabola Center, Treez of Lyfe]","[the Parabola Center, Treez of Lyfe]","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100]","[017, 017]","[Cambridge, Cambridge]"
3785,00000179-5b82-df8c-ad7d-db8231bd0001,Wednesday May 12,NOVA explores barriers to fertility from the s...,None,"[NOVA, NOVA, Shutterstock, bezikus Eky Studio]","[GBH 2, GBH 2]","[GBH, GBH, PBS]","[NOVA, Shutterstock, bezikus Eky Studio, GBH 2...","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100, 365100, 365100]","[017, 017, 017, 017]","[Cambridge, Cambridge, Cambridge, Cambridge]"
...,...,...,...,...,...,...,...,...,...,...,...,...
3745,00000179-4703-d2db-af7b-770b44830001,Make Mom Happy With These Wines According To L...,Nothing says Mother Day quite like brunch so t...,None,"[Boston Nath lie Wine Bar, NV, Fortier]","[Nath lie Wine Bar, Fortier, Fortier, Nath lie...",[Valfaccenda Vindabeive],"[Boston Nath lie Wine Bar, NV, Fortier, Valfac...","[[-71.0588801, 42.3600825], [-71.3824374, 42.4...","[030302, 365100, 365100]","[025, 017, 017]","[Downtown, Cambridge, Cambridge]"
1057,00000176-7635-d997-af77-777596910001,Saturday December 19,Nothing warms wintry weekend like busy kitchen...,None,"[GBH 2, WGBH]","[GBH 2, GBH 2, Greater Boston Hermitage 2]","[GBH, WGBH, WGBH, America Test Kitchen, Great ...","[GBH 2, WGBH, Greater Boston Hermitage 2, Amer...","[[-71.3824374, 42.4072107], [-71.1172973, 42.3...","[365100, 353900, 365100, 365100, 365100]","[017, 017, 017, 017, 017]","[Cambridge, Cambridge, Cambridge, Cambridge, C..."
3874,00000179-7d84-df8c-ad7d-fd86f5860001,Eric In The Evening Sunday May 16 2021,00000179 7d84 df8c ad7d fd86f5860002,None,None,[Canal Street],None,[Canal Street],[],[],[],[]
10921,00000184-56e6-d8c1-a99d-57e6e8a20002,Six races for governor that could make history...,U.S. politicians are supposed to represent the...,None,[Rutgers University Eagleton Institute of Poli...,"[Rutgers University, Eagleton Institute of Pol...","[Rutgers University, Eagleton Institute of Pol...",[Rutgers University Eagleton Institute of Poli...,"[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100, 365100, 353102, 365100]","[017, 017, 017, 017, 017]","[Cambridge, Cambridge, Cambridge, Cambridge, C..."


In [124]:
df.to_csv(f"./results/full_benchmark_{sample_count}_samples_trial_{trial}.csv")
time_df.to_csv(f"./results/full_benchmark_times_{sample_count}_samples_trial_{trial}.csv")